## Question: "How faint a source can I detect at N-sigma in a fixed exposure time?"

### Some NIRCam Docs
- [Readout Patterns](https://jwst-docs.stsci.edu/jwst-near-infrared-camera/nircam-instrumentation/nircam-detector-overview/nircam-detector-readout-patterns#gsc.tab=0)
- [Recommended Strategies](https://jwst-docs.stsci.edu/jwst-near-infrared-camera/nircam-observing-strategies/nircam-imaging-recommended-strategies#gsc.tab=0)

In [2]:
# Pandeia reference data
import os
os.environ["pandeia_refdata"] = "../../data/pandeia_data-2026.7-jwst"
os.environ["PSF_DIR"] = "../../data/pandeia_psfs-2026.7rc1-jwst"
os.environ["PYSYN_CDBS"] = "../../data/grp/redcat/trds"

# Exposure time calculator
from pandeia.engine.calc_utils import build_default_calc
from pandeia.engine.perform_calculation import perform_calculation

# Telescope
telescope = 'jwst'
instrument = 'nircam'
mode = 'lw_imaging'
filter = 'f356w'
readout_pattern = "medium8" # "deep2", "deep8", "medium8", "shallow4", "bright2", "rapid"

# Fixed exposure recipe -- these alone set the total exposure time (no tuning loop)
ngroup = 9 # 10 max for all, except deep2/deep8 = 20 max, rapid = 2 max
nint = 6   # 10 max allowed
nexp = 4

# Target parameters
target_snr = 5.0 # 5-sigma depth

# Flux parameters
flux_unit = 'abmag' # flux unit
mag_bright = 20.0 # brightest bound of the depth search (AB mag)
mag_faint = 35.0  # faintest bound of the depth search (AB mag)

# 1) Set default calculation
calc = build_default_calc(telescope, instrument, mode)

# 2) Set instrument/filter and point-source normalization in AB for the same bandpass
calc['configuration']['instrument']['filter'] = filter
calc['scene'][0]['shape']['geometry'] = 'point'
calc['scene'][0]['spectrum']['normalization'] = {
    'type': f'{telescope}',
    'bandpass': f'{instrument},{mode},{filter}',
    'norm_flux': 0.5 * (mag_bright + mag_faint),  # placeholder: the solver overwrites this every iteration
    'norm_fluxunit': flux_unit
}

# 3) Detector settings (fixed)
calc['configuration']['detector']['readout_pattern'] = readout_pattern  # pandeia uses lowercase names
calc['configuration']['detector']['ngroup'] = ngroup
calc['configuration']['detector']['nint'] = nint
calc['configuration']['detector']['nexp'] = nexp

# 4) Background level
calc['background_level'] = 'low'  # 'low', 'medium', 'high'

In [3]:
def solve_ab_depth(calc, target_snr, low, high, max_iter=30, tol=0.02):
    """
    Find AB magnitude where S/N ~= target_snr via bisection.
    low/high bracket the search (bright/faint AB mag).
    Returns (depth_ab, final_snr_at_depth)
    """
    lo0, hi0 = low, high
    check_mag = snr = None

    # Bisection method
    for _ in range(max_iter):
        check_mag = 0.5 * (low + high)

        # At this mag, what's the S/N?
        calc['scene'][0]['spectrum']['normalization']['norm_flux'] = check_mag
        snr = perform_calculation(calc)['scalar']['sn']

        # Adjust bounds
        if snr >= target_snr:
            low = check_mag # Too bright
        else:
            high = check_mag # Too faint

        # Return if S/N close
        if abs(snr - target_snr) < tol:
            break
    else:
        print(f"[warn] no convergence in {max_iter} iterations (S/N={snr:.2f})")

    # The depth must sit inside the bracket, not pinned to an edge
    if min(abs(check_mag - lo0), abs(check_mag - hi0)) < 0.05:
        print(f"[warn] depth {check_mag:.2f} is at the edge of the search range "
              f"[{lo0}, {hi0}] -- widen mag_bright/mag_faint")

    return check_mag, snr

# 5) Report the fixed exposure
rep = perform_calculation(calc)
exp = rep['information']['exposure_specification']
single_int = exp['exposure_time'] / exp['nint']

# 6) Solve for AB depth at target S/N
depth_ab, sn_at_depth = solve_ab_depth(calc, target_snr, mag_bright, mag_faint)

print(f"\n=== {telescope}/{instrument}: {target_snr}sig depth @ {exp['total_exposure_time']:.0f} s ===")
print(f"Depth: {depth_ab:.3f} AB")
print(f"S/N at depth: {sn_at_depth:.2f}")
print(f"Total exposure time: {exp['total_exposure_time']:.2f} seconds "
      f"(ngroup={exp['ngroup']}, nint={exp['nint']}, nexp={exp['nexp']}, readout={exp['readout_pattern']})")
print(f"single integration = {single_int:.2f} seconds")


=== jwst/nircam: 5.0sig depth @ 22891 s ===
Depth: 29.558 AB
S/N at depth: 5.01
Total exposure time: 22890.79 seconds (ngroup=9, nint=6, nexp=4, readout=medium8)
single integration = 953.78 seconds
